# 22 — Does a learned visual JEPA help choose actions?

**Lesson type: small-scale local experiment.** This lesson trains compact models on rendered MountainCar transitions. It asks whether predicting future visual representations helps a planner select actions.

It is not a reproduction of V-JEPA 2, LeJEPA, V-JEPA 2.1, RoboJEPA, or a robot benchmark. The simulator renderer is synthetic. The models are trained from scratch on one task.

**Learning outcomes**

- Separate one-step prediction, changed-action prediction, and closed-loop task outcomes.
- Compare a pixel-reconstructing latent model with a latent-prediction JEPA objective.
- See how a planner can exploit model errors.
- Inspect the cost and observation access of each method.
- Repeat the study on CPU or, when supported, CUDA.


## Fixed protocol

Notebook 22 has three run modes. `pilot` runs one development seed on the fixed train/validation split and writes a validation-only record with a hash of the training code. It does not collect final intervention or episode cases. `full` skips final model training and all final cases unless the latest matching pilot passed. `smoke` checks the pipeline on reserved smoke seeds; its results are not a comparison.

| Part | Full-run protocol |
|---|---|
| Task | Gymnasium `MountainCarContinuous-v0`, goal position 0.45, 300-step limit |
| Visual context | Two rendered frames, resized to 48 × 128 RGB |
| Training | 20 random-action episodes per data group; first 16 train, last 4 validate |
| Training-data seeds | 5100, 5101, 5102, 5103, 5104 |
| Model seeds | 7221, 7227, 7239, 7253, 7263 |
| Intervention seeds | 2100–2104 |
| Closed-loop episode seeds | 3100–3119 |
| Interventions | 20 held-out starts × actions {-1, 0, 1} |
| Closed-loop | 20 shared starts for each method and each learned-model seed |
| CEM | Horizon 10, 24 candidates, 4 elites, 2 updates, replan after each observed step |
| Objective | Each model scores plans in its own state, pixel, or latent space |
| Device | A synchronized CUDA operation must pass; otherwise use CPU |

Pilot mode uses training data/model seeds `1000/2021`. Smoke mode uses `9000/9021`, intervention seed `2900`, and episode seeds `3900/3901`. A pre-gate exploratory run used training pairs `2121/1100`, `2127/1101`, `2139/1102`, `2153/1103`, and `2163/1104`; these seeds are permanently excluded from the final comparison. Earlier smoke outputs with overlapping draft seeds are also excluded.

Use `JEPA_RUN_MODE=pilot` for a validation-only training run. After a matching pilot passes the predeclared collapse screen, restart the notebook with `JEPA_RUN_MODE=full`. The full mode requires the same training-code hash, objective settings, seeds, split, and device as the passing pilot. Set `JEPA_RUN_OUTPUT_DIR` to an E: or D: folder to keep run outputs there.

**Current status (2026-10-15):** the fixed development pilot failed the collapse screen. Five pre-gate training runs also collapsed and are excluded from the final comparison. The replacement final seeds and all final intervention/evaluation cases remain unused. See the [run report](../docs/reproductions/visual-jepa-decision-usefulness.md).


In [ ]:
import csv
import hashlib
import json
import os
import platform
import sys
import tempfile
import time
import warnings
from pathlib import Path

import gymnasium as gym
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL
import scipy
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

from jepa_tutorial.action_conditioned_visual import (
    ActionConditionedJEPA,
    ImageReconstructionDynamics,
    StateDynamics,
    embedding_collapse_diagnostics,
    select_device,
)
from jepa_tutorial.lejepa_loss import lejepa_total_loss
from jepa_tutorial.paper_losses import epps_pulley_sigreg_loss

RUN_MODE = os.environ.get("JEPA_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "pilot", "smoke"}:
    raise ValueError("JEPA_RUN_MODE must be 'full', 'pilot', or 'smoke'")

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
OUTPUT_DIR = Path(os.environ.get("JEPA_RUN_OUTPUT_DIR", ROOT / "docs" / "evidence"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX = {"full": "visual-jepa", "pilot": "visual-jepa-pilot", "smoke": "visual-jepa-smoke"}[RUN_MODE]

DEVICE = select_device()
DEVICE_NAME = (
    torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor() or "CPU"
)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

GOAL_POSITION = 0.45
MIN_POSITION = -1.2
MAX_POSITION = 0.6
MAX_SPEED = 0.07
MAX_STEPS = 300
IMAGE_WIDTH = 256
IMAGE_HEIGHT = 96
RESIZED_SIZE = (128, 48)
ACTIONS = (-1.0, 0.0, 1.0)
CEM_HORIZON = 10
CEM_CANDIDATES = 24
CEM_ELITES = 4
CEM_UPDATES = 2
BATCH_SIZE = 64 if RUN_MODE == "smoke" else 256
EPOCHS = int(os.environ.get("JEPA_EPOCHS", "2")) if RUN_MODE == "smoke" else 35
SIGREG_WEIGHT = 0.1 if RUN_MODE == "full" else float(os.environ.get("JEPA_SIGREG_WEIGHT", "0.1"))
SIGREG_PROJECTIONS = 128
TRAIN_EPISODES = 6 if RUN_MODE == "smoke" else 20
TRAIN_SPLIT_EPISODES = 4 if RUN_MODE == "smoke" else 16
MODEL_SEEDS = {
    "full": (7221, 7227, 7239, 7253, 7263),
    "pilot": (2021,),
    "smoke": (9021,),
}[RUN_MODE]
TRAIN_DATA_SEEDS = {
    "full": (5100, 5101, 5102, 5103, 5104),
    "pilot": (1000,),
    "smoke": (9000,),
}[RUN_MODE]
EVALUATION_SEEDS = tuple(range(3100, 3120)) if RUN_MODE == "full" else ((3900, 3901) if RUN_MODE == "smoke" else ())
INTERVENTION_SEEDS = tuple(range(2100, 2105)) if RUN_MODE == "full" else ((2900,) if RUN_MODE == "smoke" else ())
INTERVENTION_STEPS = (40, 90, 140, 190)
NUM_INTERVENTION_STARTS = 20 if RUN_MODE == "full" else (4 if RUN_MODE == "smoke" else 0)
PILOT_RECORD_PATH = ROOT / "docs" / "evidence" / "visual-jepa-validation-pilots.csv"
NOTEBOOK_PATH = ROOT / "notebooks" / "22_visual_jepa_decision_usefulness.ipynb"
TRAINING_SOURCE_PATHS = (
    ROOT / "src" / "jepa_tutorial" / "action_conditioned_visual.py",
    ROOT / "src" / "jepa_tutorial" / "paper_losses.py",
    ROOT / "src" / "jepa_tutorial" / "lejepa_loss.py",
)


def training_protocol_hash():
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    digest = hashlib.sha256()
    for index in (2, 3, 5, 7):
        digest.update("".join(notebook["cells"][index]["source"]).encode("utf-8"))
    for source_path in TRAINING_SOURCE_PATHS:
        digest.update(source_path.read_bytes())
    return digest.hexdigest()


TRAINING_PROTOCOL_HASH = training_protocol_hash()


def locked_validation_pilot_passed(path, protocol_hash):
    required = {
        "selection_stage": "training-validation-only",
        "encoder_pool": "average",
        "layernorm_affine": "false",
        "variance_weight": "0",
        "learning_rate": "0.002",
        "epochs": "35",
        "training_episodes": "16",
        "validation_episodes": "4",
        "train_rows": "4800",
        "validation_rows": "1200",
        "model_seed": "2021",
        "data_seed": "1000",
        "device": str(DEVICE),
        "sigreg_weight": "0.1",
        "evaluation_data_used": "false",
        "protocol_hash": protocol_hash,
    }
    if not path.is_file():
        return False
    with path.open(newline="", encoding="utf-8") as handle:
        matching_rows = [
            row for row in csv.DictReader(handle)
            if all(row.get(field, "").strip().lower() == value.lower() for field, value in required.items())
        ]
    return bool(matching_rows) and matching_rows[-1].get("collapsed", "true").strip().lower() == "false"


LOCKED_PILOT_PASSED = locked_validation_pilot_passed(PILOT_RECORD_PATH, TRAINING_PROTOCOL_HASH)
RUN_FINAL_TRAINING = RUN_MODE in {"pilot", "smoke"} or (RUN_MODE == "full" and LOCKED_PILOT_PASSED)
RUN_FINAL_EVALUATION = RUN_MODE == "smoke" or (RUN_MODE == "full" and LOCKED_PILOT_PASSED)


torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(2021)
np.random.seed(2021)

print("Run mode:", RUN_MODE)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("Selected device:", DEVICE, DEVICE_NAME)
print("Precision: float32")
print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)
print("Matplotlib:", matplotlib.__version__)
print("SciPy:", scipy.__version__)
print("Platform:", platform.platform())
print("Model seeds:", MODEL_SEEDS)
print("Training data seeds:", TRAIN_DATA_SEEDS)
print("Evaluation seeds:", EVALUATION_SEEDS)
print("Training episodes per group:", TRAIN_EPISODES)
print("Epochs:", EPOCHS)
print("SIGReg weight/projections:", SIGREG_WEIGHT, SIGREG_PROJECTIONS)
print("Training protocol hash:", TRAINING_PROTOCOL_HASH)
print("Locked validation pilot passed:", LOCKED_PILOT_PASSED)
print("Final model training enabled:", RUN_FINAL_TRAINING)
print("Final intervention/evaluation enabled:", RUN_FINAL_EVALUATION)
print("Output directory:", OUTPUT_DIR)


In [ ]:
X_MIN_PIXEL = 8
X_MAX_PIXEL = IMAGE_WIDTH - 8
X_PIXELS, Y_PIXELS = np.meshgrid(
    np.arange(IMAGE_WIDTH, dtype=np.float64),
    np.arange(IMAGE_HEIGHT, dtype=np.float64),
)


def position_to_pixel(position):
    fraction = (float(position) - MIN_POSITION) / (MAX_POSITION - MIN_POSITION)
    return X_MIN_PIXEL + np.clip(fraction, 0.0, 1.0) * (X_MAX_PIXEL - X_MIN_PIXEL)


def pixel_to_position(pixel, image_width=IMAGE_WIDTH):
    left = X_MIN_PIXEL * image_width / IMAGE_WIDTH
    right = X_MAX_PIXEL * image_width / IMAGE_WIDTH
    fraction = (float(pixel) - left) / (right - left)
    return MIN_POSITION + np.clip(fraction, 0.0, 1.0) * (MAX_POSITION - MIN_POSITION)


def track_height(position):
    return int(70 - 20 * np.sin(3.0 * float(position)))


_background = Image.new("RGB", (IMAGE_WIDTH, IMAGE_HEIGHT), (18, 24, 34))
_background_draw = ImageDraw.Draw(_background)
_track_points = [
    (x, track_height(pixel_to_position(x)))
    for x in range(X_MIN_PIXEL, X_MAX_PIXEL + 1, 2)
]
_background_draw.line(_track_points, fill=(150, 160, 170), width=2)
STATIC_BACKGROUND = np.asarray(_background, dtype=np.uint8)


def render_frame(position):
    frame = STATIC_BACKGROUND.copy()
    car_x = position_to_pixel(position)
    car_y = track_height(position) - 7
    goal_x = position_to_pixel(GOAL_POSITION)
    goal_y = track_height(GOAL_POSITION) - 18
    car_blob = np.exp(-0.5 * (
        ((X_PIXELS - car_x) / 1.7) ** 2 + ((Y_PIXELS - car_y) / 1.7) ** 2
    ))
    goal_blob = np.exp(-0.5 * (
        ((X_PIXELS - goal_x) / 1.7) ** 2 + ((Y_PIXELS - goal_y) / 1.7) ** 2
    ))
    frame[..., 0] = np.clip(
        frame[..., 0].astype(np.float64) + 230 * car_blob, 0, 255
    ).astype(np.uint8)
    frame[..., 1] = np.clip(
        frame[..., 1].astype(np.float64) + 230 * goal_blob, 0, 255
    ).astype(np.uint8)
    return frame


def resize_frame(frame):
    return np.asarray(
        Image.fromarray(frame).resize(RESIZED_SIZE, resample=Image.Resampling.BILINEAR),
        dtype=np.uint8,
    )


def color_center_x(frame, color):
    red, green, blue = (frame[..., i].astype(np.float64) for i in range(3))
    if color == "red":
        weights = np.maximum(red - np.maximum(green, blue), 0.0)
    elif color == "green":
        weights = np.maximum(green - np.maximum(red, blue), 0.0)
    else:
        raise ValueError("color must be red or green")
    column_weights = weights.sum(axis=0)
    total = float(column_weights.sum())
    if total <= 0:
        raise ValueError(f"No {color} marker in rendered frame")
    return float(np.dot(np.arange(frame.shape[1]), column_weights) / total)


def visual_state(previous_frame, current_frame):
    position = pixel_to_position(
        color_center_x(current_frame, "red"), current_frame.shape[1]
    )
    previous_position = pixel_to_position(
        color_center_x(previous_frame, "red"), previous_frame.shape[1]
    )
    velocity = float(np.clip(position - previous_position, -MAX_SPEED, MAX_SPEED))
    return np.asarray([position, velocity], dtype=np.float32)


def stack_context(previous_frame, current_frame):
    return np.concatenate((previous_frame, current_frame), axis=2)


def image_tensor(context):
    return torch.as_tensor(
        np.ascontiguousarray(context.transpose(2, 0, 1)),
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).div_(255.0)


def make_env():
    env = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_STEPS)
    env.unwrapped.goal_position = GOAL_POSITION
    return env

example = render_frame(-0.5)
plt.figure(figsize=(8, 2))
plt.imshow(example)
plt.axis("off")
plt.title("Synthetic MountainCar frame; the red marker is the car and green is the goal")
plt.show()


## Collect transitions with an episode-level split

Each sample stores three consecutive frames. The model observes frames `t-1` and `t`, then predicts the context at `t+1`. This gives the model a visual cue for velocity.

The final four episodes in each data group are validation data. Interventions and closed-loop starts use other environment seeds.


In [ ]:
def collect_training_group(data_seed):
    env = make_env()
    frame_triplets = []
    actions = []
    states = []
    next_states = []
    episode_ids = []
    for episode_id in range(TRAIN_EPISODES):
        seed = data_seed * 100 + episode_id
        observation, _ = env.reset(seed=seed)
        previous_frame = resize_frame(render_frame(float(observation[0])))
        current_frame = previous_frame
        rng = np.random.default_rng(seed + 50000)
        for step in range(MAX_STEPS):
            state = np.asarray(env.unwrapped.state, dtype=np.float32).copy()
            action = float(rng.uniform(-1.0, 1.0))
            next_observation, _, terminated, truncated, _ = env.step(
                np.asarray([action], dtype=np.float32)
            )
            next_state = np.asarray(env.unwrapped.state, dtype=np.float32).copy()
            next_frame = resize_frame(render_frame(float(next_observation[0])))
            frame_triplets.append(np.stack([
                frame.transpose(2, 0, 1)
                for frame in (previous_frame, current_frame, next_frame)
            ]))
            actions.append(action)
            states.append(state)
            next_states.append(next_state)
            episode_ids.append(episode_id)
            previous_frame, current_frame = current_frame, next_frame
            if terminated or truncated:
                break
        if (episode_id + 1) % 5 == 0:
            print(f"  collected {episode_id + 1}/{TRAIN_EPISODES} episodes")
    env.close()
    return {
        "frames": np.stack(frame_triplets),
        "actions": np.asarray(actions, dtype=np.float32),
        "states": np.stack(states).astype(np.float32),
        "next_states": np.stack(next_states).astype(np.float32),
        "episode_ids": np.asarray(episode_ids, dtype=np.int32),
        "data_seed": data_seed,
    }


def collect_intervention_starts():
    env = make_env()
    starts = []
    for seed in INTERVENTION_SEEDS:
        observation, _ = env.reset(seed=seed)
        previous_frame = resize_frame(render_frame(float(observation[0])))
        current_frame = previous_frame
        rng = np.random.default_rng(seed + 70000)
        wanted = set(INTERVENTION_STEPS)
        for step in range(max(INTERVENTION_STEPS) + 1):
            if step in wanted and len(starts) < NUM_INTERVENTION_STARTS:
                starts.append({
                    "start_id": f"{seed}-{step}",
                    "seed": seed,
                    "state": np.asarray(env.unwrapped.state, dtype=np.float32).copy(),
                    "previous_frame": previous_frame.copy(),
                    "current_frame": current_frame.copy(),
                })
            action = float(rng.uniform(-1.0, 1.0))
            next_observation, _, terminated, truncated, _ = env.step(
                np.asarray([action], dtype=np.float32)
            )
            next_frame = resize_frame(render_frame(float(next_observation[0])))
            previous_frame, current_frame = current_frame, next_frame
            if terminated or truncated:
                break
    env.close()
    if len(starts) != NUM_INTERVENTION_STARTS:
        raise RuntimeError(
            f"Expected {NUM_INTERVENTION_STARTS} held-out starts, got {len(starts)}"
        )
    return starts



## Train three dynamics models

The privileged state model receives simulator position and velocity. The image-reconstruction model predicts a future embedding and decodes a future frame. The JEPA predicts the next two-frame embedding without a pixel decoder.

This lesson uses predictive MSE plus the project's Epps–Pulley SIGReg implementation. It is a local LeJEPA-inspired objective, not the paper's full training recipe. Its collapse screen is fixed before evaluation: flag non-finite embeddings or more than half of validation dimensions with standard deviation below `1e-3`. This is a local diagnostic, not a universal threshold. The validation pilot failed this screen, so no decision-quality claim is available yet.


In [ ]:
def make_batch(group, indices):
    frames = torch.as_tensor(group["frames"][indices], device=DEVICE)
    previous, current, next_frame = frames[:, 0], frames[:, 1], frames[:, 2]
    context = torch.cat((previous, current), dim=1).float().div_(255.0)
    target_context = torch.cat((current, next_frame), dim=1).float().div_(255.0)
    next_rgb = next_frame.float().div_(255.0)
    actions = torch.as_tensor(group["actions"][indices], device=DEVICE)
    states = torch.as_tensor(group["states"][indices], device=DEVICE)
    next_states = torch.as_tensor(group["next_states"][indices], device=DEVICE)
    return context, target_context, next_rgb, actions, states, next_states


def model_loss(model, method, batch, sigreg_generator=None):
    context, target_context, next_rgb, actions, states, next_states = batch
    if method == "state_model":
        prediction = model(states, actions)
        return F.mse_loss(prediction, next_states), {}
    if method == "image_reconstruction":
        prediction, image_prediction = model(context, actions)
        target = model.encoder(target_context)
        prediction_loss = F.mse_loss(prediction, target)
        pixel_loss = F.l1_loss(image_prediction, next_rgb)
        return prediction_loss + 0.25 * pixel_loss, {
            "prediction_loss": prediction_loss,
            "pixel_loss": pixel_loss,
        }
    prediction, target = model(context, actions, target_context)
    current = model.encoder(context)
    sigreg = epps_pulley_sigreg_loss(
        torch.cat((current, target), dim=0),
        num_projections=SIGREG_PROJECTIONS,
        num_points=17,
        generator=sigreg_generator,
    )
    total = lejepa_total_loss(prediction, target, sigreg, lambda_sigreg=SIGREG_WEIGHT)
    return total, {"prediction_loss": F.mse_loss(prediction, target), "sigreg": sigreg}


def validation_metrics(model, method, group, indices, batch_size):
    model.eval()
    totals = {}
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            batch = make_batch(group, batch_indices)
            loss, components = model_loss(model, method, batch)
            totals.setdefault("total_loss", []).append(float(loss.item()))
            for name, value in components.items():
                totals.setdefault(name, []).append(float(value.item()))
            if method == "jepa":
                embeddings.append(model.encoder(batch[1]))
    metrics = {name: float(np.mean(values)) for name, values in totals.items()}
    if method == "jepa":
        diagnostics = embedding_collapse_diagnostics(
            torch.cat(embeddings, dim=0), std_threshold=1e-3, collapsed_fraction=0.5
        )
        metrics.update(diagnostics)
    return metrics


def fit_one_model(group, method, model_seed, epochs=EPOCHS):
    torch.manual_seed(model_seed)
    model = {
        "state_model": StateDynamics,
        "image_reconstruction": ImageReconstructionDynamics,
        "jepa": ActionConditionedJEPA,
    }[method]().to(DEVICE)
    train_indices = np.flatnonzero(group["episode_ids"] < TRAIN_SPLIT_EPISODES)
    validation_indices = np.flatnonzero(group["episode_ids"] >= TRAIN_SPLIT_EPISODES)
    if not len(train_indices) or not len(validation_indices):
        raise ValueError("Episode split must contain both training and validation rows")
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    rng = np.random.default_rng(model_seed)
    sigreg_generator = torch.Generator(device=DEVICE).manual_seed(model_seed + 1)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model.train()
    for epoch in range(epochs):
        order = rng.permutation(train_indices)
        for offset in range(0, len(order), BATCH_SIZE):
            batch_indices = order[offset : offset + BATCH_SIZE]
            batch = make_batch(group, batch_indices)
            optimizer.zero_grad(set_to_none=True)
            loss, _ = model_loss(model, method, batch, sigreg_generator)
            if not bool(torch.isfinite(loss).item()):
                raise FloatingPointError(f"{method} produced a non-finite loss")
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0 or epoch + 1 == epochs:
            print(f"    {method}: epoch {epoch + 1}/{epochs}")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    train_seconds = time.perf_counter() - started
    peak_mib = (
        float(torch.cuda.max_memory_allocated() / 1024**2)
        if DEVICE.type == "cuda"
        else None
    )
    metrics = validation_metrics(
        model, method, group, validation_indices, batch_size=BATCH_SIZE
    )
    record = {
        "method": method,
        "model_seed": model_seed,
        "training_data_seed": group["data_seed"],
        "status": "complete",
        "device": str(DEVICE),
        "device_name": DEVICE_NAME,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda or "",
        "precision": "float32",
        "sigreg_weight": SIGREG_WEIGHT if method == "jepa" else "",
        "sigreg_projections": SIGREG_PROJECTIONS if method == "jepa" else "",
        "epochs": epochs,
        "train_rows": len(train_indices),
        "validation_rows": len(validation_indices),
        "train_seconds": train_seconds,
        "peak_allocated_mib": peak_mib,
        **metrics,
        "error": "",
    }
    model.eval()
    return model, record


METHODS_TO_TRAIN = ("jepa",) if RUN_MODE == "pilot" else ("state_model", "image_reconstruction", "jepa")
models = {method: {} for method in METHODS_TO_TRAIN}
model_run_records = []
training_pairs_to_run = zip(TRAIN_DATA_SEEDS, MODEL_SEEDS) if RUN_FINAL_TRAINING else ()
for data_seed, model_seed in training_pairs_to_run:
    print(f"Data group {data_seed}; model seed {model_seed}")
    group = collect_training_group(data_seed)
    for method in METHODS_TO_TRAIN:
        try:
            trained_model, record = fit_one_model(group, method, model_seed)
            models[method][model_seed] = trained_model
            model_run_records.append(record)
            print(
                f"  {method}: validation loss={record['total_loss']:.6f}; "
                f"train seconds={record['train_seconds']:.1f}; "
                f"peak MiB={record['peak_allocated_mib']}"
            )
        except Exception as exc:
            model_run_records.append({
                "method": method,
                "model_seed": model_seed,
                "training_data_seed": data_seed,
                "status": "error",
                "device": str(DEVICE),
                "device_name": DEVICE_NAME,
                "torch_version": torch.__version__,
                "cuda_version": torch.version.cuda or "",
                "precision": "float32",
                "sigreg_weight": SIGREG_WEIGHT if method == "jepa" else "",
                "sigreg_projections": SIGREG_PROJECTIONS if method == "jepa" else "",
                "epochs": EPOCHS,
                "train_rows": "",
                "validation_rows": "",
                "train_seconds": "",
                "peak_allocated_mib": "",
                "total_loss": "",
                "prediction_loss": "",
                "pixel_loss": "",
                "sigreg": "",
                "finite": "",
                "mean_std": "",
                "low_variance_dims": "",
                "dimension": "",
                "collapsed": "",
                "error": repr(exc),
            })
            print(f"  {method}: FAILED: {exc}")
    del group

model_fields = [
    "method", "model_seed", "training_data_seed", "status", "device", "device_name",
    "torch_version", "cuda_version", "precision", "sigreg_weight",
    "sigreg_projections", "epochs", "train_rows",
    "validation_rows", "train_seconds", "peak_allocated_mib", "total_loss",
    "prediction_loss", "pixel_loss", "sigreg", "finite", "mean_std",
    "low_variance_dims", "dimension", "collapsed", "error",
]
model_path = OUTPUT_DIR / f"{FILE_PREFIX}-model-runs.csv"
if RUN_FINAL_TRAINING:
    existing_rows = []
    if RUN_MODE == "full" and model_path.is_file():
        with model_path.open(newline="", encoding="utf-8") as handle:
            existing_rows = list(csv.DictReader(handle))
    new_keys = {(r["method"], str(r["model_seed"]), str(r["training_data_seed"])) for r in model_run_records}
    preserved_rows = [
        r for r in existing_rows
        if (r.get("method", ""), r.get("model_seed", ""), r.get("training_data_seed", "")) not in new_keys
    ]
    with model_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=model_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(preserved_rows + model_run_records)
    print("Model-run records:", model_path)
else:
    print("Final model training was skipped because the locked validation pilot did not pass.")
print("Model runs:", len(model_run_records), "complete:", sum(r["status"] == "complete" for r in model_run_records))

if RUN_MODE == "pilot" and model_run_records and model_run_records[0]["status"] == "complete":
    record = model_run_records[0]
    pilot_fields = [
        "run_id", "selection_stage", "encoder_pool", "layernorm_affine", "sigreg_weight",
        "variance_weight", "learning_rate", "epochs", "training_episodes", "validation_episodes",
        "train_rows", "validation_rows", "model_seed", "data_seed", "device", "gpu",
        "torch_version", "cuda_version", "train_seconds", "peak_allocated_mib",
        "validation_prediction_mse", "validation_sigreg", "validation_embedding_mean_std",
        "low_variance_dimensions", "embedding_dimensions", "collapsed", "evaluation_data_used",
        "notes", "protocol_hash",
    ]
    pilot_row = {
        "run_id": "notebook-pilot-1000-2021",
        "selection_stage": "training-validation-only",
        "encoder_pool": "average",
        "layernorm_affine": "false",
        "sigreg_weight": SIGREG_WEIGHT,
        "variance_weight": 0,
        "learning_rate": 0.002,
        "epochs": EPOCHS,
        "training_episodes": TRAIN_SPLIT_EPISODES,
        "validation_episodes": TRAIN_EPISODES - TRAIN_SPLIT_EPISODES,
        "train_rows": record["train_rows"],
        "validation_rows": record["validation_rows"],
        "model_seed": record["model_seed"],
        "data_seed": record["training_data_seed"],
        "device": record["device"],
        "gpu": record["device_name"],
        "torch_version": record["torch_version"],
        "cuda_version": record["cuda_version"],
        "train_seconds": record["train_seconds"],
        "peak_allocated_mib": record["peak_allocated_mib"],
        "validation_prediction_mse": record["prediction_loss"],
        "validation_sigreg": record["sigreg"],
        "validation_embedding_mean_std": record["mean_std"],
        "low_variance_dimensions": record["low_variance_dims"],
        "embedding_dimensions": record["dimension"],
        "collapsed": record["collapsed"],
        "evaluation_data_used": "false",
        "notes": "Notebook pilot mode; no final intervention or episode seeds were collected.",
        "protocol_hash": TRAINING_PROTOCOL_HASH,
    }
    pilot_rows = []
    if PILOT_RECORD_PATH.is_file():
        with PILOT_RECORD_PATH.open(newline="", encoding="utf-8") as handle:
            pilot_rows = list(csv.DictReader(handle))
    prior_ids = {row.get("run_id", "") for row in pilot_rows}
    base_id = pilot_row["run_id"]
    run_id = base_id
    suffix = 2
    while run_id in prior_ids:
        run_id = f"{base_id}-{suffix}"
        suffix += 1
    pilot_row["run_id"] = run_id
    fd, temp_name = tempfile.mkstemp(prefix="visual-jepa-pilots-", suffix=".csv", dir=PILOT_RECORD_PATH.parent)
    os.close(fd)
    try:
        with open(temp_name, "w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=pilot_fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(pilot_rows)
            writer.writerow(pilot_row)
        os.replace(temp_name, PILOT_RECORD_PATH)
    except BaseException:
        Path(temp_name).unlink(missing_ok=True)
        raise
    print("Validation-only pilot record:", PILOT_RECORD_PATH, "run:", run_id)

if RUN_FINAL_EVALUATION:
    heldout_starts = collect_intervention_starts()
    print("Held-out intervention starts:", len(heldout_starts))
    print("Actions per start:", ACTIONS)
else:
    heldout_starts = []
    if RUN_MODE == "pilot":
        print("Pilot mode collects no final intervention or episode cases.")
    else:
        print(
            "Final intervention and episode cases remain closed because the latest "
            "protocol-matched validation-only pilot did not pass."
        )


## Test changed actions from the same starts

Each start is evaluated under three actions. The simulator state is used to record the actual next position and to score the privileged baselines. Image methods receive only the two-frame context.

Each model compares predicted with observed endpoint scores in its own space: simulator state, pixels, or latent features. These scores have different units. Latent or pixel score agreement is not task-reward accuracy. Action-ranking accuracy is reported within each model and training seed.


In [ ]:
from jepa_tutorial.evidence_validation import validate_intervention_records


PREDICTION_SPACES = {
    "oracle": "simulator_state",
    "state_model": "simulator_state",
    "image_reconstruction": "pixels_and_latent",
    "jepa": "latent",
}
SCORING_SPACES = {
    "oracle": "simulator_state_goal_distance",
    "state_model": "simulator_state_goal_distance",
    "image_reconstruction": "pixel_goal_l1",
    "jepa": "latent_goal_mse",
}


def context_tensor(previous_frame, current_frame):
    return image_tensor(stack_context(previous_frame, current_frame))


GOAL_FRAME = resize_frame(render_frame(GOAL_POSITION))
GOAL_CONTEXT = stack_context(GOAL_FRAME, GOAL_FRAME)
GOAL_RGB = torch.as_tensor(
    GOAL_FRAME.transpose(2, 0, 1).copy(), dtype=torch.float32, device=DEVICE
).unsqueeze(0).div_(255.0)
GOAL_LATENT_CACHE = {}


def goal_assets(model=None):
    if model is None:
        return GOAL_FRAME, GOAL_RGB, None
    key = id(model)
    if key not in GOAL_LATENT_CACHE:
        GOAL_LATENT_CACHE[key] = model.encode(image_tensor(GOAL_CONTEXT)).detach()
    return GOAL_FRAME, GOAL_RGB, GOAL_LATENT_CACHE[key]


def actual_step_from_state(env, state, action, reset_seed):
    env.reset(seed=reset_seed)
    env.unwrapped.state = np.asarray(state, dtype=np.float32).copy()
    next_observation, reward, terminated, truncated, _ = env.step(
        np.asarray([action], dtype=np.float32)
    )
    next_state = np.asarray(env.unwrapped.state, dtype=np.float32).copy()
    next_frame = resize_frame(render_frame(float(next_observation[0])))
    return next_state, next_frame, float(reward), bool(terminated), bool(truncated)


def rank_pair_counts(rows):
    correct = 0
    total = 0
    by_start = {}
    for row in rows:
        by_start.setdefault((row["method"], row["model_seed"], row["start_id"]), []).append(row)
    for case_rows in by_start.values():
        for i in range(len(case_rows)):
            for j in range(i + 1, len(case_rows)):
                actual_delta = case_rows[i]["observed_score"] - case_rows[j]["observed_score"]
                predicted_delta = case_rows[i]["predicted_score"] - case_rows[j]["predicted_score"]
                if abs(actual_delta) < 1e-8:
                    continue
                total += 1
                correct += int(actual_delta * predicted_delta > 0)
    return correct, total


intervention_records = []
intervention_env = make_env()
for start in heldout_starts:
    context = context_tensor(start["previous_frame"], start["current_frame"])
    for action in ACTIONS:
        actual_state, next_frame, _, _, _ = actual_step_from_state(
            intervention_env, start["state"], action, start["seed"] + 90000
        )
        target_context = stack_context(start["current_frame"], next_frame)
        observed_score = -abs(float(actual_state[0]) - GOAL_POSITION)
        intervention_records.append({
            "method": "oracle",
            "prediction_space": PREDICTION_SPACES["oracle"],
            "scoring_space": SCORING_SPACES["oracle"],
            "device": "cpu",
            "model_seed": "",
            "start_id": start["start_id"],
            "action": action,
            "actual_position": float(actual_state[0]),
            "predicted_position": float(actual_state[0]),
            "position_abs_error": 0.0,
            "velocity_abs_error": 0.0,
            "pixel_l1_error": "",
            "latent_mse": "",
            "observed_score": observed_score,
            "predicted_score": observed_score,
            "status": "complete",
            "error": "",
        })
        state_tensor = torch.as_tensor(
            start["state"], dtype=torch.float32, device=DEVICE
        ).unsqueeze(0)
        target_context_tensor = image_tensor(target_context)
        next_rgb = torch.as_tensor(
            next_frame.transpose(2, 0, 1).copy(),
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(0).div_(255.0)
        action_tensor = torch.tensor([action], dtype=torch.float32, device=DEVICE)
        for model_seed in MODEL_SEEDS:
            for method in METHODS_TO_TRAIN:
                model = models[method].get(model_seed)
                if model is None:
                    intervention_records.append({
                        "method": method,
                        "prediction_space": PREDICTION_SPACES[method],
                        "scoring_space": SCORING_SPACES[method],
                        "device": str(DEVICE),
                        "model_seed": model_seed,
                        "start_id": start["start_id"],
                        "action": action,
                        "actual_position": float(actual_state[0]),
                        "predicted_position": "",
                        "position_abs_error": "",
                        "velocity_abs_error": "",
                        "pixel_l1_error": "",
                        "latent_mse": "",
                        "observed_score": observed_score,
                        "predicted_score": "",
                        "status": "missing_model",
                        "error": "",
                    })
                    continue
                method_observed_score = observed_score
                position_abs_error = ""
                velocity_abs_error = ""
                pixel_l1_error = ""
                latent_mse = ""
                try:
                    model.eval()
                    with torch.no_grad():
                        if method == "state_model":
                            predicted_state = model(state_tensor, action_tensor)[0].cpu().numpy()
                            pred_position = float(predicted_state[0])
                            position_abs_error = abs(pred_position - float(actual_state[0]))
                            velocity_abs_error = abs(float(predicted_state[1]) - float(actual_state[1]))
                            predicted_score = -abs(pred_position - GOAL_POSITION)
                        elif method == "image_reconstruction":
                            pred_latent, pred_rgb = model(context, action_tensor)
                            target_latent = model.encoder(target_context_tensor)
                            latent_mse = float(F.mse_loss(pred_latent, target_latent).item())
                            pixel_l1_error = float(F.l1_loss(pred_rgb, next_rgb).item())
                            _, goal_rgb, _ = goal_assets()
                            pred_position = ""
                            predicted_score = -float(F.l1_loss(pred_rgb, goal_rgb).item())
                            method_observed_score = -float(F.l1_loss(next_rgb, goal_rgb).item())
                        else:
                            pred_latent = model.predict_embedding(context, action_tensor)
                            target_latent = model.encode(target_context_tensor)
                            _, _, goal_latent = goal_assets(model)
                            latent_mse = float(F.mse_loss(pred_latent, target_latent).item())
                            pred_position = ""
                            predicted_score = -float(F.mse_loss(pred_latent, goal_latent).item())
                            method_observed_score = -float(F.mse_loss(target_latent, goal_latent).item())
                    intervention_records.append({
                        "method": method,
                        "prediction_space": PREDICTION_SPACES[method],
                        "scoring_space": SCORING_SPACES[method],
                        "device": str(DEVICE),
                        "model_seed": model_seed,
                        "start_id": start["start_id"],
                        "action": action,
                        "actual_position": float(actual_state[0]),
                        "predicted_position": pred_position,
                        "position_abs_error": position_abs_error,
                        "velocity_abs_error": velocity_abs_error,
                        "pixel_l1_error": pixel_l1_error,
                        "latent_mse": latent_mse,
                        "observed_score": method_observed_score,
                        "predicted_score": predicted_score,
                        "status": "complete",
                        "error": "",
                    })
                except Exception as exc:
                    intervention_records.append({
                        "method": method,
                        "prediction_space": PREDICTION_SPACES[method],
                        "scoring_space": SCORING_SPACES[method],
                        "device": str(DEVICE),
                        "model_seed": model_seed,
                        "start_id": start["start_id"],
                        "action": action,
                        "actual_position": float(actual_state[0]),
                        "predicted_position": "",
                        "position_abs_error": "",
                        "velocity_abs_error": "",
                        "pixel_l1_error": "",
                        "latent_mse": "",
                        "observed_score": observed_score,
                        "predicted_score": "",
                        "status": "error",
                        "error": repr(exc),
                    })
intervention_env.close()

intervention_fields = [
    "method", "prediction_space", "scoring_space", "device", "model_seed",
    "start_id", "action", "actual_position",
    "predicted_position", "position_abs_error", "velocity_abs_error",
    "pixel_l1_error", "latent_mse", "observed_score", "predicted_score",
    "status", "error",
]
validate_intervention_records(intervention_records)
intervention_path = OUTPUT_DIR / f"{FILE_PREFIX}-interventions.csv"
with intervention_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=intervention_fields)
    writer.writeheader()
    writer.writerows(intervention_records)

print("Intervention records:", intervention_path)
print("Held-out starts:", len(heldout_starts), "actions per start:", len(ACTIONS))
print("Rows:", len(intervention_records))
for method in ("oracle", *METHODS_TO_TRAIN):
    rows = [r for r in intervention_records if r["method"] == method and r["status"] == "complete"]
    correct, total = rank_pair_counts(rows)
    print(f"{method}: {correct}/{total} non-tied within-model action pairs ranked correctly")


## Plan in latent or state space, then check the real episode

CEM repeatedly samples action sequences, scores the predicted endpoint, and executes only the first action. The episode then supplies a new observed state or image context.

The uniform planner starts from a zero-action mean. The policy-guided planner starts around a scripted momentum action inferred from the two input frames. This is a hand-written prior, not a learned policy. The oracle and state model read simulator state; their results are privileged diagnostics.


In [ ]:
def exact_mountaincar_step(states, actions):
    position = states[:, 0]
    velocity = states[:, 1]
    action = actions.reshape(-1)
    velocity = velocity + action * 0.0015 - 0.0025 * torch.cos(3.0 * position)
    velocity = velocity.clamp(-MAX_SPEED, MAX_SPEED)
    position = (position + velocity).clamp(MIN_POSITION, MAX_POSITION)
    velocity = torch.where(position <= MIN_POSITION, torch.zeros_like(velocity), velocity)
    return torch.stack((position, velocity), dim=1)


def score_candidate_sequences(method, model, state, context, action_sequences):
    candidate_count, horizon = action_sequences.shape
    actions = torch.as_tensor(
        action_sequences, dtype=torch.float32, device=DEVICE
    )
    with torch.no_grad():
        if method in {"oracle_state_cem", "state_model_cem"}:
            states = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
            states = states.unsqueeze(0).expand(candidate_count, -1).clone()
            for t in range(horizon):
                step_actions = actions[:, t]
                if method == "oracle_state_cem":
                    states = exact_mountaincar_step(states, step_actions)
                else:
                    states = model(states, step_actions)
                    states = states.clamp(
                        min=torch.tensor([MIN_POSITION, -MAX_SPEED], device=DEVICE),
                        max=torch.tensor([MAX_POSITION, MAX_SPEED], device=DEVICE),
                    )
            position_error = states[:, 0] - GOAL_POSITION
            score = -(position_error.square() + 0.2 * states[:, 1].square())
            return score.cpu().numpy()

        latent = model.encode(context).expand(candidate_count, -1).clone()
        for t in range(horizon):
            latent = model.predictor(latent, actions[:, t])
        if method == "image_reconstruction_cem":
            predicted_frame = model.decoder(latent)
            _, goal_rgb, _ = goal_assets()
            score = -F.l1_loss(
                predicted_frame, goal_rgb.expand_as(predicted_frame), reduction="none"
            ).mean(dim=(1, 2, 3))
        else:
            _, _, goal_latent = goal_assets(model)
            score = -F.mse_loss(
                latent, goal_latent.expand_as(latent), reduction="none"
            ).mean(dim=1)
        return score.cpu().numpy()


def cem_action(method, model, state, previous_frame, current_frame, rng):
    context = context_tensor(previous_frame, current_frame)
    if method == "jepa_momentum_prior_cem":
        prior_action = 1.0 if visual_state(previous_frame, current_frame)[1] >= 0 else -1.0
        mean = np.full(CEM_HORIZON, prior_action, dtype=np.float64)
        std = np.full(CEM_HORIZON, 0.5, dtype=np.float64)
        score_method = "jepa_cem"
    else:
        mean = np.zeros(CEM_HORIZON, dtype=np.float64)
        std = np.ones(CEM_HORIZON, dtype=np.float64)
        score_method = method
    calls = 0
    for _ in range(CEM_UPDATES):
        candidates = np.clip(
            mean[None, :] + rng.normal(size=(CEM_CANDIDATES, CEM_HORIZON)) * std[None, :],
            -1.0,
            1.0,
        )
        scores = score_candidate_sequences(
            score_method, model, state, context, candidates
        )
        elite_indices = np.argsort(scores)[-CEM_ELITES:]
        elites = candidates[elite_indices]
        mean = elites.mean(axis=0)
        std = np.maximum(elites.std(axis=0), 0.05)
        calls += CEM_CANDIDATES * CEM_HORIZON
    return float(np.clip(mean[0], -1.0, 1.0)), calls


def run_episode(method, model, model_seed, eval_seed):
    env = make_env()
    observation, _ = env.reset(seed=eval_seed)
    previous_frame = resize_frame(render_frame(float(observation[0])))
    current_frame = previous_frame
    rng = np.random.default_rng(eval_seed + model_seed * 1009)
    total_return = 0.0
    model_calls = 0
    planning_seconds = 0.0
    status = "complete"
    error = ""
    success = False
    steps = 0
    started = time.perf_counter()
    try:
        for step in range(MAX_STEPS):
            state = np.asarray(env.unwrapped.state, dtype=np.float32).copy()
            if method == "direct_visual_momentum":
                action = float(1.0 if visual_state(previous_frame, current_frame)[1] >= 0 else -1.0)
            else:
                planning_started = time.perf_counter()
                if method == "oracle_state_cem":
                    action, calls = cem_action(
                        method, None, state, previous_frame, current_frame, rng
                    )
                elif method == "state_model_cem":
                    action, calls = cem_action(
                        method, model, state, previous_frame, current_frame, rng
                    )
                elif method == "image_reconstruction_cem":
                    action, calls = cem_action(
                        method, model, state, previous_frame, current_frame, rng
                    )
                elif method in {"jepa_uniform_cem", "jepa_momentum_prior_cem"}:
                    action, calls = cem_action(
                        method, model, state, previous_frame, current_frame, rng
                    )
                else:
                    raise ValueError(f"Unknown method: {method}")
                planning_seconds += time.perf_counter() - planning_started
                model_calls += calls
            next_observation, reward, terminated, truncated, _ = env.step(
                np.asarray([action], dtype=np.float32)
            )
            total_return += float(reward)
            steps = step + 1
            next_frame = resize_frame(render_frame(float(next_observation[0])))
            previous_frame, current_frame = current_frame, next_frame
            if terminated:
                success = True
                break
            if truncated:
                break
    except Exception as exc:
        status = "error"
        error = repr(exc)
    elapsed = time.perf_counter() - started
    env.close()
    return {
        "method": method,
        "model_seed": model_seed if model_seed is not None else "",
        "eval_seed": eval_seed,
        "status": status,
        "success": int(success),
        "return": total_return,
        "steps": steps,
        "model_calls": model_calls,
        "planning_seconds": planning_seconds,
        "elapsed_seconds": elapsed,
        "device": str(DEVICE),
        "error": error,
    }


In [ ]:
evaluation_records = []
evaluation_seeds_to_run = EVALUATION_SEEDS if RUN_FINAL_EVALUATION else ()
baseline_methods = ("oracle_state_cem", "direct_visual_momentum")
for method in baseline_methods:
    for eval_seed in evaluation_seeds_to_run:
        evaluation_records.append(run_episode(method, None, 0, eval_seed))

for model_seed in MODEL_SEEDS:
    for eval_seed in evaluation_seeds_to_run:
        for method, store_key in (
            ("state_model_cem", "state_model"),
            ("image_reconstruction_cem", "image_reconstruction"),
            ("jepa_uniform_cem", "jepa"),
            ("jepa_momentum_prior_cem", "jepa"),
        ):
            model = models[store_key].get(model_seed)
            if model is None:
                evaluation_records.append({
                    "method": method,
                    "model_seed": model_seed,
                    "eval_seed": eval_seed,
                    "status": "missing_model",
                    "success": 0,
                    "return": "",
                    "steps": 0,
                    "model_calls": 0,
                    "planning_seconds": "",
                    "elapsed_seconds": "",
                    "device": str(DEVICE),
                    "error": "",
                })
                continue
            model.eval()
            evaluation_records.append(
                run_episode(method, model, model_seed, eval_seed)
            )
        if eval_seed == EVALUATION_SEEDS[-1]:
            print(f"Completed evaluation seed {eval_seed} for model seed {model_seed}")

episode_fields = [
    "method", "model_seed", "eval_seed", "status", "success", "return", "steps",
    "model_calls", "planning_seconds", "elapsed_seconds", "device", "error",
]
episode_path = OUTPUT_DIR / f"{FILE_PREFIX}-episodes.csv"
with episode_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=episode_fields)
    writer.writeheader()
    writer.writerows(evaluation_records)

print("Episode records:", episode_path)
print("Rows:", len(evaluation_records))


## Results: keep training runs separate from evaluation starts

For trained methods, the summary first computes one score per model seed. Evaluation episodes from one model do not count as independent training runs. Direct control and oracle have no training seed. Their intervals describe only the fixed evaluation starts.

The comparisons below are descriptive for this synthetic MountainCar task. Differences in state, pixel, and latent scoring spaces matter when interpreting the planner outcomes. A low prediction loss from a collapsed model is not evidence of useful representation learning. The 2026-10-15 validation pilot is recorded separately; the full intervention and closed-loop comparison remains pending.


In [ ]:
def mean_t_interval(values):
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 0:
        return float("nan"), float("nan"), float("nan")
    if len(values) == 1:
        return float(values[0]), float("nan"), float("nan")
    from scipy.stats import t
    mean = float(values.mean())
    half = float(t.ppf(0.975, df=len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values)))
    return mean, mean - half, mean + half


def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return float("nan"), float("nan")
    rate = successes / total
    denominator = 1 + z * z / total
    center = (rate + z * z / (2 * total)) / denominator
    margin = z * np.sqrt(rate * (1 - rate) / total + z * z / (4 * total * total)) / denominator
    return center - margin, center + margin


trained_methods = {
    "state_model_cem", "image_reconstruction_cem",
    "jepa_uniform_cem", "jepa_momentum_prior_cem",
}
method_order = [
    "oracle_state_cem", "state_model_cem", "image_reconstruction_cem",
    "jepa_uniform_cem", "jepa_momentum_prior_cem", "direct_visual_momentum",
]
method_labels = {
    "oracle_state_cem": "Oracle state CEM",
    "state_model_cem": "Learned state CEM",
    "image_reconstruction_cem": "Image reconstruction CEM",
    "jepa_uniform_cem": "JEPA uniform CEM",
    "jepa_momentum_prior_cem": "JEPA momentum-prior CEM",
    "direct_visual_momentum": "Direct visual momentum",
}

summary_rows = []
for method in method_order:
    complete = [r for r in evaluation_records if r["method"] == method and r["status"] == "complete"]
    if method in trained_methods:
        per_seed = []
        for model_seed in MODEL_SEEDS:
            seed_rows = [r for r in complete if r["model_seed"] == model_seed]
            if seed_rows:
                per_seed.append({
                    "model_seed": model_seed,
                    "episodes": len(seed_rows),
                    "success_rate": float(np.mean([r["success"] for r in seed_rows])),
                    "mean_return": float(np.mean([r["return"] for r in seed_rows])),
                    "mean_steps": float(np.mean([r["steps"] for r in seed_rows])),
                })
        success_mean, success_low, success_high = mean_t_interval(
            [r["success_rate"] for r in per_seed]
        )
        return_mean, return_low, return_high = mean_t_interval(
            [r["mean_return"] for r in per_seed]
        )
        summary_rows.append({
            "method": method,
            "label": method_labels[method],
            "complete_episodes": len(complete),
            "model_runs": len(per_seed),
            "success_rate": success_mean,
            "success_low": success_low,
            "success_high": success_high,
            "mean_return": return_mean,
            "return_low": return_low,
            "return_high": return_high,
            "mean_steps": float(np.mean([r["mean_steps"] for r in per_seed])) if per_seed else float("nan"),
        })
        print(
            f"{method_labels[method]}: seed-mean success={success_mean:.3f} "
            f"(95% t interval {success_low:.3f}–{success_high:.3f}; "
            f"n={len(per_seed)} model seeds), mean return={return_mean:.2f}"
        )
        for row in per_seed:
            print(
                f"  seed {row['model_seed']}: success={row['success_rate']:.3f}, "
                f"return={row['mean_return']:.2f}, steps={row['mean_steps']:.1f}"
            )
    else:
        successes = sum(r["success"] for r in complete)
        rate = successes / len(complete) if complete else float("nan")
        low, high = wilson_interval(successes, len(complete))
        mean_return = float(np.mean([r["return"] for r in complete])) if complete else float("nan")
        summary_rows.append({
            "method": method,
            "label": method_labels[method],
            "complete_episodes": len(complete),
            "model_runs": 0,
            "success_rate": rate,
            "success_low": low,
            "success_high": high,
            "mean_return": mean_return,
            "return_low": float("nan"),
            "return_high": float("nan"),
            "mean_steps": float(np.mean([r["steps"] for r in complete])) if complete else float("nan"),
        })
        print(
            f"{method_labels[method]}: {successes}/{len(complete)} starts, "
            f"success={rate:.3f} (95% Wilson interval {low:.3f}–{high:.3f}), "
            f"mean return={mean_return:.2f}"
        )

direct_by_seed = {
    r["eval_seed"]: r for r in evaluation_records
    if r["method"] == "direct_visual_momentum" and r["status"] == "complete"
}
for method in sorted(trained_methods):
    differences = []
    for row in evaluation_records:
        if row["method"] != method or row["status"] != "complete":
            continue
        direct = direct_by_seed.get(row["eval_seed"])
        if direct is not None:
            differences.append(float(row["return"]) - float(direct["return"]))
    if differences:
        print(
            f"Paired {method} minus direct control: mean return delta "
            f"{np.mean(differences):.2f} across {len(differences)} matched "
            "model-seed/evaluation-start rows; descriptive, not an independent sample."
        )

failed = [r for r in evaluation_records if r["status"] != "complete"]
print("Evaluation rows:", len(evaluation_records), "complete:", len(evaluation_records) - len(failed))
print("Failed or missing rows:", len(failed))


In [ ]:
# Figure 1: decision outcome. Trained-method intervals use training-seed variation;
# the direct and oracle intervals use fixed evaluation-start variation.
fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(summary_rows))
rates = np.asarray([row["success_rate"] for row in summary_rows])
lower = np.asarray([row["success_rate"] - row["success_low"] for row in summary_rows])
upper = np.asarray([row["success_high"] - row["success_rate"] for row in summary_rows])
colors = ["#475569", "#64748b", "#2563eb", "#7c3aed", "#a855f7", "#16a34a"]
ax.bar(x, rates, color=colors, edgecolor="#1f2937", linewidth=0.6)
if np.isfinite(np.stack((lower, upper))).all():
    ax.errorbar(x, rates, yerr=np.stack((lower, upper)), fmt="none", color="#111827", capsize=4)
ax.set_xticks(x, [row["label"] for row in summary_rows], rotation=22, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Goal success rate")
ax.set_title("Local MountainCar outcomes; error bars use different variation sources")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
figure1 = OUTPUT_DIR / f"{FILE_PREFIX}-success.png"
fig.savefig(figure1, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", figure1)

# Figure 2: action ranking agreement from paired interventions.
rank_rows = []
for method in METHODS_TO_TRAIN:
    for model_seed in MODEL_SEEDS:
        subset = [
            r for r in intervention_records
            if r["method"] == method
            and r["model_seed"] == model_seed
            and r["status"] == "complete"
        ]
        correct, total = rank_pair_counts(subset)
        rank_rows.append((method, model_seed, correct / total if total else float("nan")))
fig, ax = plt.subplots(figsize=(8, 4.2))
for index, method in enumerate(METHODS_TO_TRAIN):
    points = [row[2] for row in rank_rows if row[0] == method and np.isfinite(row[2])]
    jitter = np.linspace(-0.08, 0.08, max(1, len(points)))
    ax.scatter(index + jitter[:len(points)], points, label=method, s=44)
    if points:
        ax.plot([index - 0.18, index + 0.18], [np.nanmean(points)] * 2, color="#111827", linewidth=2)
rank_labels = {
    "state_model": "Privileged state model",
    "image_reconstruction": "Image reconstruction",
    "jepa": "Action-conditioned JEPA",
}
ax.set_xticks(range(len(METHODS_TO_TRAIN)), [rank_labels[name] for name in METHODS_TO_TRAIN], rotation=12, ha="right")
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("Non-tied action-pair ranking accuracy")
ax.set_title("Held-out action interventions by training seed")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
figure2 = OUTPUT_DIR / f"{FILE_PREFIX}-intervention-ranking.png"
fig.savefig(figure2, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", figure2)

if RUN_MODE == "full":
    status = (
        "comparison"
        if all(r["status"] == "complete" for r in model_run_records)
        and not failed
        and all(r["status"] == "complete" for r in intervention_records)
        and len(heldout_starts) >= 20
        and all(r["collapsed"] is not True for r in model_run_records if r["method"] == "jepa")
        else "partial"
    )
elif RUN_MODE == "pilot":
    status = "validation-only-pilot"
else:
    status = "smoke"
print("Evidence status:", status)
print("Model-run evidence:", model_path)
print("Intervention evidence:", intervention_path)
print("Episode evidence:", episode_path)


## Interpretation and limits

A low one-step error does not show that the model preserves the counterfactual effect of an action. Correct action ranking does not show that a planner will succeed after repeated replanning. Closed-loop success does not show a benefit over direct control when compute or observation access differs.

**Method labels**

- **Oracle state CEM** uses exact MountainCar equations and privileged simulator state.
- **Learned state CEM** trains on simulator state and has privileged observations.
- **Image reconstruction CEM** predicts and decodes future frames.
- **JEPA CEM** predicts future visual embeddings without a pixel decoder.
- **JEPA momentum-prior CEM** uses the same JEPA model and a scripted action prior.
- **Direct visual momentum** is a hand-written color-moment controller, not a learned policy.

The methods use different observation and scoring spaces. Treat success, return, prediction error, intervention ranking, model calls, and wall time as separate evidence. This local study cannot establish robot transfer, a general JEPA planning advantage, a scaling law, or any paper result.

## Research sources

- [V-JEPA 2](https://arxiv.org/abs/2506.09985) and its [official code](https://github.com/facebookresearch/vjepa2)
- [LeJEPA](https://arxiv.org/abs/2511.08544)
- [VICReg](https://arxiv.org/abs/2105.04906)
- [VISReg](https://arxiv.org/abs/2606.02572)
- [V-JEPA 2.1](https://arxiv.org/abs/2603.14482)
- [RoboJEPA](https://arxiv.org/abs/2610.10515) and its [official code](https://github.com/facebookresearch/robo_jepa)
- Project [evaluation protocol](../docs/EVALUATION_PROTOCOL.md)

## Check your understanding

1. What does a held-out action intervention test that random-action next-frame error does not?
2. Why can the JEPA's latent goal distance fail to match true task utility?
3. Which methods see simulator state that an image-only controller cannot see?
4. What additional evidence would be needed before calling this a paper reproduction?
